# Vector and Tool Memory

Two engineering patterns that back long-term agent memory, built on LangChain + LangGraph.

The notebook builds two memory backends side-by-side: an unstructured **vector memory** for prose (notes, runbooks, observations) and a structured **tool memory** for records (orders, users, exact lookups).

## Setup

In [1]:
# !pip install -q langchain langchain-google-genai langchain-openai langchain-anthropic langchain-chroma langgraph langchain-community

from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from typing import List, Optional, Dict, Any
import os, json, sqlite3, shutil

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# os.environ['GEMINI_API_KEY']  # the variable for API key; OR
# os.environ['OPENAI_API_KEY']  # the variable for API key

llm   = init_chat_model('gpt-4o-mini', model_provider='openai', temperature=0)
embed = init_embeddings('sentence-transformers/all-MiniLM-L6-v2', provider='huggingface')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Vector memory with `InMemoryVectorStore`

Now let's build the first backend. Vector memory is how an agent remembers unstructured prose, notes it took, observations it made, runbook paragraphs it should follow.

Now, we will
- define a `VectorMemory` class that wraps any LangChain vector store behind two methods: `remember(text, meta)` and `recall(query, k)`,
- instantiate it on top of an `InMemoryVectorStore` so the demo has zero external deps,
- write 9 prose notes (the kind of thing an agent would itself jot down),
- recall them with semantic queries whose wording differs from the stored notes.

In [3]:
class VectorMemory:
    """A thin agent-facing contract over any LangChain VectorStore.

    The point of wrapping is that the agent should not know (or care) WHICH store is
    underneath -- swapping InMemoryVectorStore for Chroma is a one-line internal change.
    `remember` is the WRITE path; `recall` is the READ path. That is the entire surface.
    """

    def __init__(self, store):
        self.store = store

    def remember(self, text: str, meta: Optional[dict] = None) -> str:
        """Append a single prose memory note. Returns the document id assigned by the store.
        Metadata is free-form; carry at minimum `type` and (if applicable) a subject id."""
        doc = Document(page_content=text, metadata=meta or {})
        ids = self.store.add_documents([doc])
        return ids[0]

    def recall(self, query: str, k: int = 3) -> List[Document]:
        """Retrieve the top-k most semantically similar notes. Cosine similarity over
        embeddings -- finds notes by MEANING, not by lexical overlap."""
        return self.store.similarity_search(query, k=k)

The two methods name the only operations that matter on any vector memory backend. Anything else (delete, update, metadata filter) is a variation on these.

In [4]:
in_memory_store = InMemoryVectorStore(embedding=embed)
vmem = VectorMemory(in_memory_store)
print('VectorMemory ready over:', vmem.store.__class__.__name__)

VectorMemory ready over: InMemoryVectorStore


Now let's seed the vector memory with notes the agent would have written during past sessions. These are deliberately phrased differently from the queries we will use later — that gap is what shows off **semantic** retrieval rather than keyword matching.

In [5]:
NOTES = [
    ('Runbook: if a charge fails with INSUFFICIENT_FUNDS, email the customer and pause the subscription instead of cancelling.',
     {'type': 'runbook', 'topic': 'payments'}),
    ('Runbook: tickets open more than 48h on Pro or Enterprise must be escalated to a senior engineer.',
     {'type': 'runbook', 'topic': 'escalation'}),
    ('Incident 2025-11-14: checkout 500s for 42 minutes due to a stale Stripe webhook secret.',
     {'type': 'incident', 'date': '2025-11-14'}),
    ('Incident 2026-01-03: search latency spike caused by a hot key in Redis; mitigation was sharding the popular-products key.',
     {'type': 'incident', 'date': '2026-01-03'}),
    ('Spec: the Pro plan includes priority support, 10 seats, SAML SSO, and usage analytics.',
     {'type': 'spec', 'topic': 'pro-plan'}),
    ('Spec: the Enterprise plan adds audit logs, a dedicated CSM, and a 99.95% SLA.',
     {'type': 'spec', 'topic': 'enterprise-plan'}),
    ('Observation: customer u-42 prefers email over SMS for all notifications.',
     {'type': 'observation', 'subject': 'u-42'}),
    ('Observation: small-team startups convert best when offered a 14-day Pro trial with onboarding office hours.',
     {'type': 'observation', 'topic': 'smb'}),
    ('Engineering note: the billing service uses idempotency keys on every write; never retry a charge without regenerating the key.',
     {'type': 'engineering-note', 'topic': 'billing'}),
]

for text, meta in NOTES:
    vmem.remember(text, meta)
print(f'remembered {len(NOTES)} notes in vector memory.')

remembered 9 notes in vector memory.


Now the read path. Note that the **query wording is not a substring** of any stored note; the matches come from embedding proximity, which is the whole reason vector memory is worth its operational cost.

In [6]:
for q in [
    'What is the best way to communicate with user u-42?',
    'How should we handle a declined card?',
    'Any past outages tied to caching?',
]:
    print(f'Q: {q}')
    for i, d in enumerate(vmem.recall(q, k=2), start=1):
        print(f"  top-{i} ({d.metadata.get('type')}): {d.page_content[:110]}")
    print()

Q: What is the best way to communicate with user u-42?
  top-1 (observation): Observation: customer u-42 prefers email over SMS for all notifications.
  top-2 (spec): Spec: the Pro plan includes priority support, 10 seats, SAML SSO, and usage analytics.

Q: How should we handle a declined card?
  top-1 (runbook): Runbook: if a charge fails with INSUFFICIENT_FUNDS, email the customer and pause the subscription instead of c
  top-2 (engineering-note): Engineering note: the billing service uses idempotency keys on every write; never retry a charge without regen

Q: Any past outages tied to caching?
  top-1 (incident): Incident 2025-11-14: checkout 500s for 42 minutes due to a stale Stripe webhook secret.
  top-2 (incident): Incident 2026-01-03: search latency spike caused by a hot key in Redis; mitigation was sharding the popular-pr



## Persisting vector memory by swapping in `Chroma`

Now let's repoint the same `VectorMemory` wrapper at **Chroma**, a disk-backed vector DB. Because every LangChain vector store implements the same interface, the wrapper does not change.

Now, we will
- build a `Chroma` store, hand it to the same `VectorMemory` class, and re-seed the notes,
- recall from the persistent store and confirm semantic retrieval still works,
- print the persistence path so the reader can see the write actually went to disk.

In [7]:
# !pip install -q chromadb langchain-chroma
from langchain_chroma import Chroma

PERSIST_DIR = './chroma_memory'
if os.path.isdir(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

chroma_store = Chroma(
    collection_name='agent_long_term_memory',
    embedding_function=embed,
    persist_directory=PERSIST_DIR,
)

vmem_persistent = VectorMemory(chroma_store)
for text, meta in NOTES:
    vmem_persistent.remember(text, meta)
print(f'persistent vector memory backed by Chroma at {PERSIST_DIR}')

persistent vector memory backed by Chroma at ./chroma_memory


Same wrapper, different store. The agent code that calls `recall(...)` is identical, that is the value of keeping `VectorMemory` as the contract.

In [8]:
hits = vmem_persistent.recall('How do we deal with a customer whose payment bounced?', k=2)
print('---- Chroma recall ----')
for d in hits:
    print(f"  ({d.metadata.get('type')}) {d.page_content[:110]}")

---- Chroma recall ----
  (runbook) Runbook: if a charge fails with INSUFFICIENT_FUNDS, email the customer and pause the subscription instead of c
  (incident) Incident 2025-11-14: checkout 500s for 42 minutes due to a stale Stripe webhook secret.


## Tool memory over a `sqlite` structured store

Now the second backend. **Tool memory** is how an agent remembers structured records: rows in a database, JSON in a file, results from an API. The agent does not embed-and-search this data; it calls a typed tool that queries it precisely.

Now, we will
- create an in-memory SQLite database with `users`, `orders`, and `tickets` tables,
- seed it with a handful of rows so we have realistic shapes to query,
- define a plain Python helper `db_query(...)` that runs parameterised SQL,
- print a couple of direct query results so the backend is visible before any LLM is involved.

In [9]:
conn = sqlite3.connect(':memory:', check_same_thread=False)
conn.executescript("""
CREATE TABLE users (
    user_id     TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    plan        TEXT NOT NULL
);
CREATE TABLE orders (
    order_id   INTEGER PRIMARY KEY,
    user_id    TEXT NOT NULL,
    total_usd  REAL NOT NULL,
    status     TEXT NOT NULL,
    placed_at  TEXT NOT NULL
);
CREATE TABLE tickets (
    ticket_id  TEXT PRIMARY KEY,
    user_id    TEXT NOT NULL,
    product    TEXT NOT NULL,
    status     TEXT NOT NULL
);
""")

conn.executemany('INSERT INTO users VALUES (?,?,?)', [
    ('u-17', 'Arjun Rao',    'pro'),
    ('u-23', 'Meera Iyer',   'free'),
    ('u-31', 'Sofia Garcia', 'enterprise'),
    ('u-42', 'Priya Nair',   'pro'),
])

conn.executemany('INSERT INTO orders VALUES (?,?,?,?,?)', [
    (1041, 'u-17', 129.50, 'paid',     '2026-03-01'),
    (1042, 'u-42', 482.00, 'paid',     '2026-03-12'),
    (1043, 'u-42',  49.00, 'refunded', '2026-03-20'),
    (1044, 'u-31', 1820.00,'paid',     '2026-04-02'),
    (1045, 'u-23',  19.00, 'pending',  '2026-04-05'),
    (1046, 'u-42',  98.00, 'paid',     '2026-04-09'),
])

conn.executemany('INSERT INTO tickets VALUES (?,?,?,?)', [
    ('t-001', 'u-42', 'billing',   'open'),
    ('t-002', 'u-42', 'search',    'open'),
    ('t-003', 'u-42', 'dashboard', 'closed'),
    ('t-004', 'u-17', 'billing',   'open'),
    ('t-005', 'u-31', 'sso',       'open'),
    ('t-006', 'u-23', 'search',    'closed'),
])
conn.commit()
print('sqlite seeded:',
      conn.execute('SELECT COUNT(*) FROM users').fetchone()[0], 'users,',
      conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0], 'orders,',
      conn.execute('SELECT COUNT(*) FROM tickets').fetchone()[0], 'tickets.')

sqlite seeded: 4 users, 6 orders, 6 tickets.


A direct sanity check before any LLM enters the loop. We want the structured backend to be obviously correct so that when the agent later picks the right tool, we can attribute success to routing and not to luck.

In [10]:
def db_query(sql: str, params: tuple = ()) -> List[Dict[str, Any]]:
    """Run parameterised SQL and return rows as dicts. Parameterisation matters even in a demo:
    if you let an LLM compose raw SQL strings, you have invited prompt-driven SQL injection."""
    cur = conn.execute(sql, params)
    cols = [c[0] for c in cur.description]
    return [dict(zip(cols, r)) for r in cur.fetchall()]

print('=== order #1042 ===')
print(db_query('SELECT * FROM orders WHERE order_id = ?', (1042,)))
print('=== open tickets for u-42 ===')
print(db_query("SELECT ticket_id, product FROM tickets WHERE user_id = ? AND status = 'open'", ('u-42',)))

=== order #1042 ===
[{'order_id': 1042, 'user_id': 'u-42', 'total_usd': 482.0, 'status': 'paid', 'placed_at': '2026-03-12'}]
=== open tickets for u-42 ===
[{'ticket_id': 't-001', 'product': 'billing'}, {'ticket_id': 't-002', 'product': 'search'}]


## Failure mode: structured questions through vector memory

Now let's deliberately use the wrong backend. We will ask vector memory a precise structured question: *'what was the total of order#1042?'*; and watch the recall return irrelevant prose chunks.

Now, we will
- recall from `vmem` for a precise lookup question and inspect the top-k chunks,
- show none of them contain the requested value (because the value lives in a SQL row, not a paragraph),
- ask the same question through the structured `db_query` and confirm the exact answer,
- print a one-line architectural takeaway so the reader internalises the split.

In [11]:
bad_q = 'What was the total of order #1042?'

print('=== vector memory recall (wrong tool) ===')
for d in vmem.recall(bad_q, k=3):
    print(f"  ({d.metadata.get('type')}) {d.page_content[:110]}")

print()
print('=== same question through structured tool memory ===')
rows = db_query('SELECT order_id, user_id, total_usd, status FROM orders WHERE order_id = ?', (1042,))
print(rows)

print()
print('---- takeaway ----')
print('Embeddings cannot count, cannot join, and cannot filter exactly.')
print('Precise lookups belong to tool memory, not vector memory.')

=== vector memory recall (wrong tool) ===
  (incident) Incident 2025-11-14: checkout 500s for 42 minutes due to a stale Stripe webhook secret.
  (spec) Spec: the Pro plan includes priority support, 10 seats, SAML SSO, and usage analytics.
  (spec) Spec: the Enterprise plan adds audit logs, a dedicated CSM, and a 99.95% SLA.

=== same question through structured tool memory ===
[{'order_id': 1042, 'user_id': 'u-42', 'total_usd': 482.0, 'status': 'paid'}]

---- takeaway ----
Embeddings cannot count, cannot join, and cannot filter exactly.
Precise lookups belong to tool memory, not vector memory.


## Tool-augmented agent with `@tool` and `bind_tools`

Now let's give the LLM both backends as tools and watch it route. We expose vector memory through `vector_recall` and structured memory through `db_lookup_order`, `db_open_tickets`, and `db_get_user`.

Now, we will
- decorate the four functions with `@tool` so LangChain auto-derives a JSON schema from their signatures,
- bind them to the LLM with `llm.bind_tools([...])`,
- invoke the tool-bound LLM on three example queries (one prose, one structured, one mixed),
- print `tool_calls` from each response so the routing decision is visible.

In [12]:
@tool
def vector_recall(query: str) -> str:
    """Search the agent's UNSTRUCTURED prose memory (runbooks, incidents, specs, observations).
    Use this for questions about policy, process, past incidents, product tiers, or anything
    that lives as a paragraph rather than a row. Returns up to 3 passages."""
    docs = vmem.recall(query, k=3)
    if not docs:
        return 'No relevant passages found.'
    return '\n---\n'.join(
        f"[{d.metadata.get('type', 'doc')}] {d.page_content}" for d in docs
    )

@tool
def db_lookup_order(order_id: int) -> Dict[str, Any]:
    """Look up an order by integer order_id. Returns a dict with order_id, user_id,
    total_usd, status, placed_at -- or an empty dict if the order does not exist."""
    rows = db_query(
        'SELECT order_id, user_id, total_usd, status, placed_at FROM orders WHERE order_id = ?',
        (order_id,),
    )
    return rows[0] if rows else {}

@tool
def db_open_tickets(user_id: str) -> List[Dict[str, Any]]:
    """Return the OPEN tickets for a given user_id (e.g. 'u-42').
    Each row carries ticket_id, user_id, product, status."""
    return db_query(
        "SELECT ticket_id, user_id, product, status FROM tickets WHERE user_id = ? AND status = 'open'",
        (user_id,),
    )

@tool
def db_get_user(user_id: str) -> Dict[str, Any]:
    """Return the user record (user_id, name, plan) for a given user_id, or {} if unknown."""
    rows = db_query('SELECT user_id, name, plan FROM users WHERE user_id = ?', (user_id,))
    return rows[0] if rows else {}

Each `@tool` carries a typed signature plus a docstring. The docstring is the LLM's documentation (a vague one routes worse than a specific one). Note how each tool encodes its own predicate (e.g. `status = 'open'`) so the LLM never composes raw SQL.

In [13]:
TOOLS = [vector_recall, db_lookup_order, db_open_tickets, db_get_user]
llm_with_tools = llm.bind_tools(TOOLS)
print('bound tools:', [t.name for t in TOOLS])

bound tools: ['vector_recall', 'db_lookup_order', 'db_open_tickets', 'db_get_user']


Now the routing demo. We invoke the tool-bound LLM directly (no graph yet, just one call) and inspect `tool_calls` on each response. The interesting thing is **which** tool the model picks per question, not the final natural-language answer.

In [14]:
SYSTEM = SystemMessage(content=(
    'You are a support-ops assistant with two memory backends.\n'
    '- vector_recall: for POLICY, RUNBOOK, INCIDENT, or PROSE questions.\n'
    '- db_lookup_order / db_open_tickets / db_get_user: for STRUCTURED facts.\n'
    'Never guess counts, totals, or statuses -- always call the right tool.'
))

QUERIES = [
    ('prose',      'What does our runbook say to do when a payment fails with insufficient funds?'),
    ('structured', 'What was the total of order 1042 and is it paid?'),
    ('mixed',      'How many open tickets does u-42 have, and does our policy say they need escalation yet?'),
]

for label, q in QUERIES:
    resp = llm_with_tools.invoke([SYSTEM, HumanMessage(content=q)])
    print(f'---- {label} ----')
    print(f'Q: {q}')
    for tc in resp.tool_calls:
        print(f"  -> tool_call {tc['name']}({tc.get('args', {})})")
    if not resp.tool_calls:
        print(f'  (no tool call) {str(resp.content)[:160]}')
    print()

---- prose ----
Q: What does our runbook say to do when a payment fails with insufficient funds?
  -> tool_call vector_recall({'query': 'payment fails insufficient funds runbook'})

---- structured ----
Q: What was the total of order 1042 and is it paid?
  -> tool_call db_lookup_order({'order_id': 1042})

---- mixed ----
Q: How many open tickets does u-42 have, and does our policy say they need escalation yet?
  -> tool_call db_open_tickets({'user_id': 'u-42'})
  -> tool_call vector_recall({'query': 'escalation policy'})



The prose query routes to `vector_recall`. The structured query routes to `db_lookup_order` with `order_id=1042`. The mixed query usually fires both `db_open_tickets` and `vector_recall`.

In [15]:
# Manually dispatch the tool calls from the structured query so the reader sees the precise answer.
structured_q = QUERIES[1][1]
resp = llm_with_tools.invoke([SYSTEM, HumanMessage(content=structured_q)])
print(f'==== structured query end-to-end ====')
print('Q:', structured_q)
for tc in resp.tool_calls:
    name = tc['name']
    args = tc.get('args', {})
    fn   = {t.name: t for t in TOOLS}[name]
    out  = fn.invoke(args)
    print(f'  -> {name}({args}) = {out}')

==== structured query end-to-end ====
Q: What was the total of order 1042 and is it paid?
  -> db_lookup_order({'order_id': 1042}) = {'order_id': 1042, 'user_id': 'u-42', 'total_usd': 482.0, 'status': 'paid', 'placed_at': '2026-03-12'}


### When to use vector memory vs tool memory
- Use **vector memory** for unstructured / semantic queries: 'what does our runbook say', 'have we seen an incident like this', 'find similar past tickets'. The answer lives in a paragraph somewhere and you want the closest one by meaning.
- Use **tool memory** for precise lookups, aggregations, joins, and freshness-critical state: 'how many open tickets for u-42', 'what was the total of order 1042', 'is the dark_mode flag on'. The answer must be computed from current rows, embeddings cannot do this.
- Use **both, bound to one agent** for mixed questions ('summarise this user's tickets against our escalation policy'). With both tool classes bound via `bind_tools`, the LLM picks per question and interleaves calls in one turn.
- Default rule of thumb: **prose -> vector, record -> tool, both -> both.**

### Common pitfalls in vector + tool memory
- Never let the LLM compose raw SQL strings. Tools should expose a fixed shape (whitelisted columns, parameterised predicates) and accept only the variables. Anything else is prompt-injection-as-SQL.
- Do not embed structured records as a workaround. Vector memory looks like it answers 'what was the order total' until it confidently fabricates a number.
- Vector memory goes stale silently. The moment a runbook is edited, every embedded chunk for it is wrong until you re-index. Either re-embed on write or expose the live source through a tool.
- Each tool round-trip is an extra LLM call (model -> tool -> model). For three tool calls per query, that is four LLM calls total. Budget tokens accordingly and prefer the smallest tool surface that answers the question.